In [ ]:
# Use this script after updating the demographics data in a local gdb but before uploading it to AGOL.
# You will use this script to take the demographics data and pivot it by gender. Once you're done with this script
# (and any others that need to be run for this data), you can then upload both the tall and the pivoted data at once.

# Background:
# The original demographic data is formatted such that each different combination of geography, year, age group, 
# gender, and race are different records. This is fine for a table (though it makes for a very long table), but
# it's way too much data to display as a joined hosted feature layer. For this reason, on the data portal, pivoted
# data is used for the map.

# Ideas for improvement:
# Bypass Excel. I'm only using it because this works well enough and I have other stuff to do.

In [ ]:
# Used for re-shaping the table
import pandas as pd

# Used for working with Excel
import openpyxl

In [ ]:
# You will need to have an ArcGIS Pro project open
aprx = arcpy.mp.ArcGISProject("CURRENT")

# In that project, you will need to have a map called "Map" (or change this name)
m = aprx.listMaps("Map")[0]

# In that map, you will need to have a table called "Demographics" (or change this name)
demoData = m.listTables("Demographics")[0]

In [ ]:
# Geodatabase location & where you want to save any Excel files
location = r"C:\Users\JohnsonN35\Local_work\Copied_from_local_work\MiTracking_Exploration"

# File name
file = "Demographics_pivoted_by_gender"

# Excel extension
ext = ".xlsx"

# Geodatabase name (this script assumes your gdb is in the same place as where you want to save any Excel files)
gdb = "MiTracking_Population_Demographics.gdb"

In [ ]:
# Create a definition query for only the all-ages and all-races records
demoDataDQ = 'Age_Group_Num = 0 And "Race" = \'All\''

#Set the definition query
demoData.definitionQuery = demoDataDQ

In [ ]:
# Create a pandas dataframe

fields = ['FIPS_Table', 'Year', 'Gender', 'Population_Estimate'] # Leaving out age group and race

df = pd.DataFrame.from_records(data = arcpy.da.SearchCursor(demoData, fields), columns = fields)

df.head()

In [ ]:
# Create a wide dataframe

df_wide = df.pivot(index = ['FIPS_Table', 'Year'], columns = ['Gender'], values = ['Population_Estimate'])

df_wide.reset_index(inplace = True)

df_wide.head()

In [ ]:
# Export to an Excel spreadsheet

name = file + "_pandas"

df_wide.to_excel(location + "/" + name + ext, sheet_name = 'Sheet1')

In [ ]:
# Clean up the Excel file (we can't use the current table structure in a feature class)

from openpyxl import load_workbook

outpath = location + "/" + name + ext
wb = load_workbook(outpath)
ws = wb.active

# Append all the estimate fields with "_Estimate"
# (i.e., if estimates go from cell D2 (4th column) - F2 (6th column), put 6 next to the break)
for index, row in enumerate(ws.rows, start = 4):
        if index > 6: break
        ws.cell(row = 2, column = index).value += '_Estimate'

# Write the values of cells B1 (2nd column) - C1 (3rd column) into cells B2 - C2
for index, row in enumerate(ws.rows, start = 2):
        if index > 3: break
        ws.cell(row = 2, column = index).value = ws.cell(row = 1, column = index).value

# Unmerge estimate cells (causes issues with deleting rows in next step)
ws.unmerge_cells('D1:F1')

# Delete rows and columns we don't need and that can't be imported into a feature class
ws.delete_cols(1)
ws.delete_rows(1)
ws.delete_rows(2) # Once we delete the first row, the original third row becomes the second row
wb.save(outpath)

In [ ]:
# Create a new table to hold the wide sheet

tableName = "Demographics_pivoted_by_gender"

arcpy.management.CreateTable(location + "/" + gdb, tableName)

In [ ]:
# Create the fields for the table; 
arcpy.management.AddFields(tableName, "\
FIPS_Table TEXT 'FIPS' 7 # #;\
Year SHORT Year # # #;\
F_Estimate LONG 'Female Population Estimate' # # #;\
M_Estimate LONG 'Male Population Estimate' # # #;\
All_Estimate LONG 'Total Population Estimate' # # #\
")

In [ ]:
# Append wide sheet to new table

spreadsheet = location + "\\" + name + ext + "\\Sheet1$"

fieldMapping = '\
    FIPS_Table "FIPS" true true false 7 Text 0 0,First,#,spreadsheet,FIPS_Table,0,254;\
    Year "Year" true true false 2 Short 0 0,First,#,spreadsheet,Year,-1,-1;\
    F_Estimate "Female Population Estimate" true true false 4 Long 0 0,First,#,spreadsheet,F_Estimate,-1,-1;\
    M_Estimate "Male Population Estimate" true true false 4 Long 0 0,First,#,spreadsheet,M_Estimate,-1,-1;\
    All_Estimate "Total Population Estimate" true true false 4 Long 0 0,First,#,spreadsheet,All_Estimate,-1,-1\
    '

arcpy.management.Append(spreadsheet, tableName, "NO_TEST", fieldMapping)